# PHASE 4: MODEL IMPLEMENTATION - BIDIRECTIONAL LSTM (BI-LSTM) [V1.2]

ARSITEKTUR BARU + HYPERPARAMETER TUNING:
- **EMBEDDING 128→256** — KAPASITAS REPRESENTASI AA 2×
- **HIDDEN LSTM 128→256** — POLA KOMPLEKS LEBIH TERTANGKAP
- **BATCHNORM1D** — STABILISASI TRAINING
- **FC 4-LAYER** 1024→512→256→128→6 — HIERARKI FITUR LEBIH DALAM
- **AUGMENTASI RATE 0.05** — LEBIH KONSERVATIF
- **LR FACTOR 0.3** — TURUN LEBIH AGGRESIF SAAT PLATEAU
- **EARLY STOPPING PATIENCE 7** — EPOK LEBIH BANYAK
- **SEED 42** — TERBUKTI OPTIMAL DARI V1.1

## 1. IMPORT LIBRARY DAN KONFIGURASI DEVICE

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix, matthews_corrcoef
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import json
import random
import shutil

# KAGGLE PATHS
BASE_DIR = '/kaggle/working'
DATA_DIR = '/kaggle/input/datasets/breyhandany/project1'
os.makedirs(f'{BASE_DIR}/models', exist_ok=True)
os.makedirs(f'{BASE_DIR}/results', exist_ok=True)
os.makedirs(f'{BASE_DIR}/figures', exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {device}")

# SEED UNTUK REPRODUCIBILITY
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 2. LOAD DATASET DARI KAGGLE

In [ ]:
# 1. LOAD DATASET
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
val_df = pd.read_csv(f'{DATA_DIR}/val.csv')
test_df = pd.read_csv(f'{DATA_DIR}/test.csv')

with open(f'{DATA_DIR}/label_mapping.json', 'r') as f:
    label_mapping = json.load(f)

num_classes = len(label_mapping)
CLASS_NAMES = [label_mapping[str(i)] for i in range(num_classes)]
print(f"Data Train: {len(train_df)}, Data Val: {len(val_df)}, Data Test: {len(test_df)}, Jumlah Kelas: {num_classes}")
print(f"Kelas: {CLASS_NAMES}")

## 3. TOKENISASI ASAM AMINO DAN DATASET DENGAN AUGMENTASI (RATE 0.05)

In [ ]:
# 2. TOKENISASI & DATASET DENGAN AUGMENTASI
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
aa_to_int = {aa: i + 1 for i, aa in enumerate(AMINO_ACIDS)}
aa_to_int['<PAD>'] = 0
VOCAB_SIZE = len(aa_to_int)
MAX_LEN = 1000

class ProteinDataset(Dataset):
    def __init__(self, sequences, labels, max_len=MAX_LEN, augment=False, augment_rate=0.05):
        self.sequences = sequences
        self.labels = labels
        self.max_len = max_len
        self.augment = augment
        self.augment_rate = augment_rate

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences.iloc[idx]
        label = self.labels.iloc[idx]
        seq_int = [aa_to_int.get(aa, 0) for aa in seq[:self.max_len]]
        if self.augment and len(seq_int) > 0 and random.random() < self.augment_rate:
            pos = random.randint(0, len(seq_int) - 1)
            orig = seq_int[pos]
            while True:
                new = random.randint(1, 20)
                if new != orig:
                    break
            seq_int[pos] = new
        if len(seq_int) < self.max_len:
            seq_int += [0] * (self.max_len - len(seq_int))
        return torch.tensor(seq_int, dtype=torch.long), torch.tensor(label, dtype=torch.long)

BATCH_SIZE = 64
train_dataset = ProteinDataset(train_df['Sequence'], train_df['Label'], max_len=MAX_LEN, augment=True, augment_rate=0.05)
val_dataset = ProteinDataset(val_df['Sequence'], val_df['Label'], max_len=MAX_LEN, augment=False)
test_dataset = ProteinDataset(test_df['Sequence'], test_df['Label'], max_len=MAX_LEN, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Batch size: {BATCH_SIZE}, Max length: {MAX_LEN}, Vocab: {VOCAB_SIZE}")
print(f"Augmentasi training: AKTIF (rate={train_dataset.augment_rate}, mutasi 1 AA random)")

## 4. DEFINISI FOCAL LOSS, LABEL SMOOTHING, DAN ARSITEKTUR LSTM (256/256/BN/4-FC)

In [ ]:
# 3. FOCAL LOSS + LABEL SMOOTHING & ARSITEKTUR LSTM
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.0, label_smoothing=0.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.reduction = reduction

    def forward(self, inputs, targets):
        n_classes = inputs.size(1)
        if self.label_smoothing > 0:
            smoothed = torch.full_like(inputs, self.label_smoothing / (n_classes - 1))
            smoothed.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)
            ce_loss = -smoothed * F.log_softmax(inputs, dim=1)
            ce_loss = ce_loss.sum(dim=1)
        else:
            ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.alpha is not None:
            alpha_weight = self.alpha[targets]
            focal_loss = alpha_weight * focal_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


class ProteinLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, num_classes):
        super(ProteinLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding_dropout = nn.Dropout(p=0.2)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers,
                            bidirectional=True, batch_first=True,
                            dropout=0.4 if num_layers > 1 else 0)
        self.global_max_pool = nn.AdaptiveMaxPool1d(1)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        fc_input_dim = hidden_dim * 2 * 2
        self.bn = nn.BatchNorm1d(fc_input_dim)
        self.fc = nn.Sequential(
            nn.Linear(fc_input_dim, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes))
        self._init_weights()

    def _init_weights(self):
        for name, param in self.lstm.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)
        for module in self.fc:
            if isinstance(module, nn.Linear):
                nn.init.kaiming_uniform_(module.weight, mode='fan_in', nonlinearity='relu')
                nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.embedding(x)
        x = self.embedding_dropout(x)
        lstm_out, _ = self.lstm(x)
        lstm_out = lstm_out.transpose(1, 2)
        max_pooled = self.global_max_pool(lstm_out).squeeze(-1)
        avg_pooled = self.global_avg_pool(lstm_out).squeeze(-1)
        pooled = torch.cat([max_pooled, avg_pooled], dim=1)
        pooled = self.bn(pooled)
        return self.fc(pooled)


EMBED_DIM = 256
HIDDEN_DIM = 256
NUM_LAYERS = 2
model = ProteinLSTM(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, num_classes).to(device)
print(model)

# Hitung total parameter
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameter: {total_params:,}")

## 5. TRAINING DENGAN ARSITEKTUR BARU DAN HYPERPARAMETER TUNING

In [ ]:
# 4. TRAINING (AR SITEKTUR BARU + REDUCELRONPLATEAU FACTOR 0.3 + PATIENCE 7)
SEEDS = [42]
EPOCHS = 50
PATIENCE = 7
all_results = []

labels_array = train_df['Label'].values
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels_array), y=labels_array)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

for seed_idx, SEED in enumerate(SEEDS):
    print(f"\n{'='*60}")
    print(f"  RUN {seed_idx+1}/{len(SEEDS)} — SEED = {SEED}")
    print(f"{'='*60}")
    set_seed(SEED)

    model = ProteinLSTM(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, num_classes).to(device)
    criterion = FocalLoss(alpha=class_weights_tensor, gamma=1.0, label_smoothing=0.1)
    optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=5e-5)

    # REDUCELRONPLATEAU DENGAN FACTOR 0.3
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=2)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss = float("inf")
    best_epoch = 0
    epochs_no_improve = 0
    early_stop = False

    for epoch in range(1, EPOCHS + 1):
        if early_stop:
            break
        epoch_start = time.time()

        # TRAINING PHASE
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        for seqs, labels in train_loader:
            seqs, labels = seqs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(seqs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * seqs.size(0)
            _, preds = torch.max(outputs, 1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

        # VALIDATION PHASE
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for seqs, labels in val_loader:
                seqs, labels = seqs.to(device), labels.to(device)
                outputs = model(seqs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * seqs.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        epoch_train_loss = train_loss / train_total
        epoch_train_acc = train_correct / train_total
        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total

        history["train_loss"].append(epoch_train_loss)
        history["train_acc"].append(epoch_train_acc)
        history["val_loss"].append(epoch_val_loss)
        history["val_acc"].append(epoch_val_acc)

        # LR STEP DENGAN VAL LOSS
        scheduler.step(epoch_val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        epoch_time = time.time() - epoch_start
        print(f"Epoch {epoch:2d}/{EPOCHS} | Waktu: {epoch_time:.0f}s | LR: {current_lr:.2e} | Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

        # EARLY STOPPING
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_epoch = epoch
            epochs_no_improve = 0
            torch.save(model.state_dict(), f'{BASE_DIR}/models/lstm_model_best.pth')
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"[Early Stopping] No improvement for {PATIENCE} epochs at epoch {epoch}")
                early_stop = True

    print(f"\n--- Seed {SEED} Selesai ---")
    print(f"Best Epoch: {best_epoch} (Val Loss: {best_val_loss:.4f})")

    # LOAD BEST MODEL AND EVALUATE ON TEST SET
    model.load_state_dict(torch.load(f'{BASE_DIR}/models/lstm_model_best.pth', weights_only=True))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for seqs, labels in test_loader:
            seqs, labels = seqs.to(device), labels.to(device)
            outputs = model(seqs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    mcc = matthews_corrcoef(all_labels, all_preds)
    all_results.append({"seed": SEED, "accuracy": acc, "f1_macro": f1, "mcc": mcc, "best_epoch": best_epoch})
    print(f">>> Seed {SEED}: Acc={acc:.4f}, F1={f1:.4f}, MCC={mcc:.4f}")

best_result = all_results[0]
print(f"\nModel terbaik: Seed {best_result['seed']} (Acc={best_result['accuracy']:.4f})")

## 6. EVALUASI METRIK DAN SIMPAN MODEL

In [ ]:
# 5. EVALUASI METRIK DAN SIMPAN MODEL
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for seqs, labels in test_loader:
        seqs = seqs.to(device)
        outputs = model(seqs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("\n--- Classification Report ---")
report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES)
print(report)

mcc = matthews_corrcoef(all_labels, all_preds)
print(f"\nMCC Score: {mcc:.4f}")

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix - LSTM v1.2')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.savefig(f'{BASE_DIR}/figures/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# SIMPAN METRIK
final_metrics = {
    'accuracy': accuracy_score(all_labels, all_preds),
    'f1_macro': f1_score(all_labels, all_preds, average='macro'),
    'mcc': mcc,
    'seed': best_result['seed'],
    'history': history
}
with open(f'{BASE_DIR}/results/lstm_metrics.json', 'w') as f:
    json.dump(final_metrics, f)

# SIMPAN BOBOT MODEL
torch.save(model.state_dict(), f'{BASE_DIR}/models/lstm_model.pth')
print(f"\nModel LSTM dan hasil evaluasi berhasil disimpan di {BASE_DIR}/")

## 7. VISUALISASI KURVA LOSS & ACCURACY

In [ ]:
# 6. VISUALISASI KURVA LOSS & ACCURACY
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss', alpha=0.7)
plt.plot(history['val_loss'], label='Val Loss', alpha=0.7)
if best_epoch:
    plt.axvline(x=best_epoch - 1, color='g', linestyle='--', alpha=0.5, label=f'Best Epoch {best_epoch}')
plt.title('Loss Curves', fontsize=13)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Acc', alpha=0.7)
plt.plot(history['val_acc'], label='Val Acc', alpha=0.7)
if best_epoch:
    plt.axvline(x=best_epoch - 1, color='g', linestyle='--', alpha=0.5, label=f'Best Epoch {best_epoch}')
plt.title('Accuracy Curves', fontsize=13)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE_DIR}/figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. KOMPRES OUTPUT UNTUK DOWNLOAD

In [ ]:
# 7. KOMPRES OUTPUT UNTUK DOWNLOAD
shutil.make_archive(f'{BASE_DIR}/lstm_v1.2_output', 'zip', BASE_DIR)
print("File siap download: /kaggle/working/lstm_v1.2_output.zip")
print("Berisi: model weights, metrics JSON, figure")